# **Notebook setup**

##**Download the code**

In [420]:
%%capture
!git clone https://github.com/google/benchmark.git
!git clone https://github.com/google/googletest.git benchmark/googletest

###**Organize the code and install**

In [421]:
%%capture
!rm -rf benchmark/build
!cmake -E make_directory "benchmark/build"
!cmake -E chdir "benchmark/build" cmake -DCMAKE_BUILD_TYPE=Release ..
!cmake --build "benchmark/build" --config Release --target install

In [757]:
%%capture
!rm -rf benchmark/googletest/build
!cmake -E make_directory "benchmark/googletest/build"
!cmake -E chdir "benchmark/googletest/build" cmake -DCMAKE_BUILD_TYPE=Release ..
!cmake --build "benchmark/googletest/build" --config Release --target install

##An integer 'p' is prime iff the only integers a that divides p are a = 1 or a = p

In [770]:
%%writefile isPrime.cpp
#include <benchmark/benchmark.h>
#include <boost/multiprecision/cpp_int.hpp>
#include <boost/random.hpp>

using namespace boost::multiprecision;
using namespace boost::random;

/*Computes a^p and checks the mod to decide if the number is probably prime or not*/
uint512_t power(uint512_t a, uint512_t p, uint512_t n, bool* isProbablyPrime) {

  uint512_t x, result;

  if (p == 0) return 1;

  x = power(a, p/2, n, isProbablyPrime);

  result = (x * x) % n;

  /* check whether x^2 mod n = 1 and x != 1, n-1 */
  if (result == 1 && x != 1 && x != n-1) *isProbablyPrime = false;
  if (p % 2 == 1) result = (a * result) % n;

  return result;
}

/*Generates a random number uniformly between start and end, following boost library specification:
https://www.boost.org/doc/libs/1_89_0/libs/multiprecision/doc/html/boost_multiprecision/tut/random.html */
uint512_t randomN(uint512_t start, uint512_t end){

  static mt19937 mt(12345);
  uniform_int_distribution<uint512_t> ui(start, end);

  return ui(mt);
}

bool isPrime(uint512_t n) {

  if(n==uint512_t(0)) return false;
  if(n<=uint512_t(3)) return true;

  bool isProbablyPrime = true;

  uint512_t a = randomN(2, n-1);

  uint512_t result = power(a, n-1, n, &isProbablyPrime);

  if (result != 1 || !isProbablyPrime) return false;

  return true;
}

static void BM_PrimalityTest(benchmark::State& state)
{

  for (auto _ : state) {
    /*Always checks the numbers of the form 2^N - 1*/
    uint512_t to_check = uint512_t(1)<<= ((state.range(0)) - 1) - 1;
    benchmark::DoNotOptimize(isPrime(to_check));
  }

  state.SetComplexityN(state.range(0));
}


BENCHMARK(BM_PrimalityTest)
->RangeMultiplier(2)
->DenseRange(1LL<<6,1LL<<9,1LL<<6)
->Complexity();


BENCHMARK_MAIN();

Overwriting isPrime.cpp


In [745]:
!g++ isPrime.cpp -O2 -std=c++11 -isystem benchmark/include -Lbenchmark/build/src -lbenchmark -lboost_random -lboost_system -lpthread -o isPrime

In [746]:
!./isPrime

2025-10-21T20:58:45+00:00
Running ./isPrime
Run on (2 X 2200 MHz CPU s)
CPU Caches:
  L1 Data 32 KiB (x1)
  L1 Instruction 32 KiB (x1)
  L2 Unified 256 KiB (x1)
  L3 Unified 56320 KiB (x1)
Load Average: 0.37, 0.56, 0.73
---------------------------------------------------------------
Benchmark                     Time             CPU   Iterations
---------------------------------------------------------------
BM_PrimalityTest/64        8388 ns         8375 ns        85304
BM_PrimalityTest/128      27869 ns        27835 ns        23244
BM_PrimalityTest/192      57979 ns        57744 ns        12425
BM_PrimalityTest/256      99021 ns        98944 ns         7116
BM_PrimalityTest/320     181752 ns       178700 ns         5220
BM_PrimalityTest/384     337103 ns       318779 ns         2234
BM_PrimalityTest/448     339471 ns       323058 ns         1980
BM_PrimalityTest/512     361850 ns       347405 ns         2203
BM_PrimalityTest_BigO       1.63 N^2        1.56 N^2  
BM_PrimalityTest_RMS 

In [769]:
%%writefile isPrime2.cpp
#include <benchmark/benchmark.h>
#include <boost/multiprecision/cpp_int.hpp>
#include <boost/random.hpp>

using namespace boost::multiprecision;
using namespace boost::random;

/*Computes a^p and checks the mod to decide if the number is probably prime or not*/
uint512_t power(uint512_t a, uint512_t p, uint512_t n, bool* isProbablyPrime) {

  uint512_t x, result;

  if (p == 0) return 1;

  x = power(a, p/2, n, isProbablyPrime);

  result = (x * x) % n;

  /* check whether x^2 mod n = 1 and x != 1, n-1 */
  if (result == 1 && x != 1 && x != n-1) *isProbablyPrime = false;
  if (p % 2 == 1) result = (a * result) % n;

  return result;
}

/*Generates a random number uniformly between start and end, following boost library specification:
https://www.boost.org/doc/libs/1_89_0/libs/multiprecision/doc/html/boost_multiprecision/tut/random.html */
uint512_t randomN(uint512_t start, uint512_t end){

  static mt19937 mt(12345);
  uniform_int_distribution<uint512_t> ui(start, end);

  return ui(mt);
}

/*Generates a random number using the number of bits specified in the range*/
uint512_t random_to_check(int bits) {

  static mt19937 mt;

  uint512_t lower_bound = uint512_t(1) << (bits - 1);

  uint512_t upper_bound = (uint512_t(1) << bits) - 1;

  uniform_int_distribution<uint512_t> dist(lower_bound, upper_bound);

  return dist(mt);
}

bool isPrime(uint512_t n) {

  if(n==uint512_t(0)) return false;
  if(n<=uint512_t(3)) return true;

  bool isProbablyPrime = true;

  uint512_t a = randomN(2, n-1);

  uint512_t result = power(a, n-1, n, &isProbablyPrime);

  if (result != 1 || !isProbablyPrime) return false;

  return true;
}

static void BM_PrimalityTest(benchmark::State& state)
{

  for (auto _ : state) {
    /*Randomly generate a number to check for each iteration*/
    uint512_t to_check = random_to_check(state.range(0));
    benchmark::DoNotOptimize(isPrime(to_check));
  }

  state.SetComplexityN(state.range(0));
}


BENCHMARK(BM_PrimalityTest)
->DenseRange((1LL<<9)-8,1LL<<9)
->Complexity();


BENCHMARK_MAIN();

Overwriting isPrime2.cpp


In [747]:
!g++ isPrime2.cpp -O2 -std=c++11 -isystem benchmark/include -Lbenchmark/build/src -lbenchmark -lboost_random -lboost_system -lpthread -o isPrime2

In [753]:
!./isPrime2

2025-10-21T21:00:29+00:00
Running ./isPrime2
Run on (2 X 2200 MHz CPU s)
CPU Caches:
  L1 Data 32 KiB (x1)
  L1 Instruction 32 KiB (x1)
  L2 Unified 256 KiB (x1)
  L3 Unified 56320 KiB (x1)
Load Average: 0.98, 0.70, 0.76
---------------------------------------------------------------
Benchmark                     Time             CPU   Iterations
---------------------------------------------------------------
BM_PrimalityTest/504     270663 ns       270325 ns         2463
BM_PrimalityTest/505     268258 ns       267898 ns         2456
BM_PrimalityTest/506     273682 ns       272577 ns         2478
BM_PrimalityTest/507     266749 ns       266572 ns         2506
BM_PrimalityTest/508     272896 ns       272627 ns         2648
BM_PrimalityTest/509     280574 ns       278228 ns         2611
BM_PrimalityTest/510     478388 ns       472218 ns         1567
BM_PrimalityTest/511     441302 ns       431353 ns         1581
BM_PrimalityTest/512     297468 ns       292567 ns         2457
BM_Primalit

In [768]:
%%writefile isPrime3.cpp
#include <benchmark/benchmark.h>
#include <boost/multiprecision/cpp_int.hpp>
#include <boost/random.hpp>

using namespace boost::multiprecision;
using namespace boost::random;

/*Computes a^p and checks the mod to decide if the number is probably prime or not*/
uint512_t power(uint512_t a, uint512_t p, uint512_t n, bool* isProbablyPrime) {

  uint512_t x, result;

  if (p == 0) return 1;

  x = power(a, p/2, n, isProbablyPrime);

  result = (x * x) % n;

  /* check whether x^2 mod n = 1 and x != 1, n-1 */
  if (result == 1 && x != 1 && x != n-1) *isProbablyPrime = false;
  if (p % 2 == 1) result = (a * result) % n;

  return result;
}

/*Generates a random number uniformly between start and end, following boost library specification:
https://www.boost.org/doc/libs/1_89_0/libs/multiprecision/doc/html/boost_multiprecision/tut/random.html */
uint512_t randomN(uint512_t start, uint512_t end){

  static mt19937 mt(12345);
  uniform_int_distribution<uint512_t> ui(start, end);

  return ui(mt);
}

/*Generates a random number using the number of bits specified in the range*/
uint512_t random_to_check(int bits) {

  static mt19937 mt;

  uint512_t lower_bound = uint512_t(1) << (bits - 1);

  uint512_t upper_bound = (uint512_t(1) << bits) - 1;

  uniform_int_distribution<uint512_t> dist(lower_bound, upper_bound);

  return dist(mt);
}

bool isPrime(uint512_t n) {

  if(n==uint512_t(0)) return false;
  if(n<=uint512_t(3)) return true;

  bool isProbablyPrime = true;

  uint512_t a = randomN(2, n-1);

  uint512_t result = power(a, n-1, n, &isProbablyPrime);

  if (result != 1 || !isProbablyPrime) return false;

  return true;
}

static void BM_PrimalityTest(benchmark::State& state)
{

  for (auto _ : state) {
    /*Randomly generate a number to check for each iteration*/
    uint512_t to_check = random_to_check(state.range(0));
    benchmark::DoNotOptimize(isPrime(to_check));
  }

  state.SetComplexityN(state.range(0));
}


BENCHMARK(BM_PrimalityTest)
->DenseRange(1LL<<6,1LL<<9,1LL<<6)
->Complexity();


BENCHMARK_MAIN();

Overwriting isPrime3.cpp


In [754]:
!g++ isPrime3.cpp -O2 -std=c++11 -isystem benchmark/include -Lbenchmark/build/src -lbenchmark -lboost_random -lboost_system -lpthread -o isPrime3

In [738]:
!./isPrime3

2025-10-21T20:52:03+00:00
Running ./isPrime3
Run on (2 X 2200 MHz CPU s)
CPU Caches:
  L1 Data 32 KiB (x1)
  L1 Instruction 32 KiB (x1)
  L2 Unified 256 KiB (x1)
  L3 Unified 56320 KiB (x1)
Load Average: 0.98, 0.91, 0.89
---------------------------------------------------------------
Benchmark                     Time             CPU   Iterations
---------------------------------------------------------------
BM_PrimalityTest/64        8510 ns         8497 ns        80698
BM_PrimalityTest/128      41380 ns        41357 ns        16534
BM_PrimalityTest/192      89743 ns        89649 ns         6900
BM_PrimalityTest/256     163246 ns       163102 ns         4426
BM_PrimalityTest/320     208132 ns       207242 ns         3443
BM_PrimalityTest/384     233763 ns       233603 ns         3104
BM_PrimalityTest/448     434140 ns       424903 ns         2110
BM_PrimalityTest/512     319209 ns       301458 ns         2366
BM_PrimalityTest_BigO      80.84 NlgN      78.88 NlgN 
BM_PrimalityTest_RMS

##Tests

In [774]:
%%writefile testing.cpp
#include <gtest/gtest.h>

#include <boost/multiprecision/cpp_int.hpp>
#include <boost/random.hpp>

using namespace boost::multiprecision;
using namespace boost::random;

/*Computes a^p and checks the mod to decide if the number is probably prime or not*/
uint512_t power(uint512_t a, uint512_t p, uint512_t n, bool* isProbablyPrime) {

  uint512_t x, result;

  if (p == 0) return 1;

  x = power(a, p/2, n, isProbablyPrime);

  result = (x * x) % n;

  /* check whether x^2 mod n = 1 and x != 1, n-1 */
  if (result == 1 && x != 1 && x != n-1) *isProbablyPrime = false;
  if (p % 2 == 1) result = (a * result) % n;

  return result;
}

/*Generates a random number uniformly between start and end, following boost library specification:
https://www.boost.org/doc/libs/1_89_0/libs/multiprecision/doc/html/boost_multiprecision/tut/random.html */
uint512_t randomN(uint512_t start, uint512_t end){

  static mt19937 mt(12345);
  uniform_int_distribution<uint512_t> ui(start, end);

  return ui(mt);
}

/*Generates a random number using the number of bits specified in the range*/
uint512_t random_to_check(int bits) {

  static mt19937 mt;

  uint512_t lower_bound = uint512_t(1) << (bits - 1);

  uint512_t upper_bound = (uint512_t(1) << bits) - 1;

  uniform_int_distribution<uint512_t> dist(lower_bound, upper_bound);

  return dist(mt);
}

bool isPrime(uint512_t n) {

  if(n==uint512_t(0)) return false;
  if(n<=uint512_t(3)) return true;

  bool isProbablyPrime = true;

  uint512_t a = randomN(2, n-1);

  uint512_t result = power(a, n-1, n, &isProbablyPrime);

  if (result != 1 || !isProbablyPrime) return false;

  return true;
}

/*Testing the consistency of the algorithm against known primes, known composites, a large prime, a large composite*/
TEST(PrimalityTest, KnownPrimes) {
  EXPECT_TRUE(isPrime(2));
  EXPECT_TRUE(isPrime(3));
  EXPECT_TRUE(isPrime(5));
  EXPECT_TRUE(isPrime(7));
  EXPECT_TRUE(isPrime(13));
}

TEST(PrimalityTest, KnownComposites) {
  EXPECT_FALSE(isPrime(4));
  EXPECT_FALSE(isPrime(9));
  EXPECT_FALSE(isPrime(15));
  EXPECT_FALSE(isPrime(21));
}

TEST(PrimalityTest, LargeKnownPrime) {
  /*Large Mersenne prime*/
  uint512_t n = (uint512_t(1) << 31) - 1;
  EXPECT_TRUE(isPrime(n));
}

TEST(PrimalityTest, LargeComposite) {
  /*Large Mersenne composite*/
  uint512_t n = (uint512_t(1) << 11) - 1; // 2047 = 23 * 89
  EXPECT_FALSE(isPrime(n));
}

TEST(PrimalityTest, EdgeCases) {
    EXPECT_FALSE(isPrime(0));
    EXPECT_TRUE(isPrime(2));
    EXPECT_TRUE(isPrime(3));
}

int main(int argc, char **argv) {
  ::testing::InitGoogleTest(&argc, argv);
  return RUN_ALL_TESTS();
}

Overwriting testing.cpp


In [775]:
!g++ testing.cpp -lgtest -lgtest_main -lboost_random -lboost_system -lpthread -o testing

In [776]:
!./testing

[==========] Running 5 tests from 1 test suite.
[----------] Global test environment set-up.
[----------] 5 tests from PrimalityTest
[ RUN      ] PrimalityTest.KnownPrimes
[       OK ] PrimalityTest.KnownPrimes (0 ms)
[ RUN      ] PrimalityTest.KnownComposites
[       OK ] PrimalityTest.KnownComposites (0 ms)
[ RUN      ] PrimalityTest.LargeKnownPrime
[       OK ] PrimalityTest.LargeKnownPrime (0 ms)
[ RUN      ] PrimalityTest.LargeComposite
[       OK ] PrimalityTest.LargeComposite (0 ms)
[ RUN      ] PrimalityTest.EdgeCases
[       OK ] PrimalityTest.EdgeCases (0 ms)
[----------] 5 tests from PrimalityTest (0 ms total)

[----------] Global test environment tear-down
[==========] 5 tests from 1 test suite ran. (0 ms total)
[  PASSED  ] 5 tests.
